# Are ROADS pALS in ECAS?

In [1]:
%load_ext autoreload
%autoreload 2

In [18]:
import pandas as pd; pd.set_option("display.max_columns", None)
import sys;sys.path.append("..")
from config import Paths
from dataframes import *
from eals_data.utils import venn_diagram
from eals_radcliff.utils import dataframes as radcliff_dataframes
from eals_radcliff.utils import dataframes_paper as radcliff_dataframes_paper
from eals_data.utils import venn_diagram 

In [ ]:
from eals_ecas import dataframes as ecas_dataframes

# ECAS
df_kely_full = ecas_dataframes.get_df_kely()
df_kely = ecas_dataframes.get_df_kely_paper()
df_latest = ecas_dataframes.get_df_latest()
df_latest.surname = df_latest.name.apply(lambda x: x.split(' ')[-1] if isinstance(x, str) else x)

[2025-11-07 17:22:48 - INFO] - User 68714bdb-5df7-494f-a917-728396faca49 has no valid survey
[2025-11-07 17:22:48 - INFO] - User 68714bdb-5df7-494f-a917-728396faca49 has no valid survey
[2025-11-07 17:22:49 - INFO] - User b9e38c66-e3bd-4e97-9ce9-225ed42548c3 has no valid survey
[2025-11-07 17:22:49 - INFO] - User b9e38c66-e3bd-4e97-9ce9-225ed42548c3 has no valid survey
[2025-11-07 17:22:49 - INFO] - User b9e38c66-e3bd-4e97-9ce9-225ed42548c3 has no valid survey
[2025-11-07 17:23:02 - INFO] - User aec1150b-f8f5-48a9-85ce-a20041ddde57 has no valid survey
[2025-11-07 17:23:02 - INFO] - User 499d65b4-2b89-42e0-a47d-94b253e31647 has no valid survey
[2025-11-07 17:23:03 - INFO] - User b0b0ee17-cb04-46df-b6a2-7a1a4cd9d1b7 has no valid survey
[2025-11-07 17:23:03 - INFO] - User d4dcf98b-35d9-42c1-823c-f971880a994f has no valid survey
[2025-11-07 17:23:03 - INFO] - User 5cff2783-107b-4905-951c-94bdf0ae53ac has no valid survey
[2025-11-07 17:23:03 - INFO] - User aced8e10-dc95-4b66-9c42-2eb0efda3d

In [13]:
df_als = load_alsfrsr_data()
print(f'N pALS: {df_als.user_id.nunique()}, N sessions: {df_als.session_id.nunique()}')

N pALS: 11, N sessions: 88


In [ ]:
# None of the ROADS pALS are in ECAS (Kely paper)
venn_diagram(df_als.user_id.unique(), df_kely.user_id.unique())

(11, 0, 56)

In [ ]:
# None of the ROADS pALS are in ECAS (Kely paper full)
venn_diagram(df_als.user_id.unique(), df_kely_full.user_id.unique())

(11, 0, 74)

### Intentar con apellido
tampoco matchea nada

In [44]:
from eals_radcliff.data_modules import sql_portal

connection = sql_portal.get_connection()

# Read participants table (contains surnames)
participants = pd.read_sql("SELECT * FROM participants", connection)
participants['participant_lname'] = participants.participant_lname.str.lower()
participants['participant_name'] = participants.participant_name.str.lower()
participants = participants[['participants_id', 'participant_name', 'participant_lname']].drop_duplicates()

# Read participants_accounts (contains eals_id)
participants_accounts = pd.read_sql("SELECT * FROM participants_accounts", connection)
participants_accounts = participants_accounts[['eals_id', 'participants_id']].drop_duplicates()

connection.close()

# Df to merge
df_data = participants.merge(participants_accounts, on='participants_id', how='inner').dropna(subset=['eals_id'])
df_data['full_name'] = df_data.participant_name + ' ' + df_data.participant_lname
df_data = df_data[['eals_id', 'participant_name', 'participant_lname', 'full_name']]
df_als = df_als.merge(df_data, left_on='user_id', right_on='eals_id', how='left')

In [45]:
venn_diagram(df_als.participant_lname.unique(), df_latest.surname.unique())

(11, 0, 118)